# 01 — Weather Data Collection

This notebook recreates the weather-data collection used in the thesis.

**Source:** Open-Meteo Historical Weather API  
**Reanalysis model:** ERA5-Land  
**Temporal resolution:** hourly  
**Time standard:** UTC  
**Period:** 2015-01-01 to 2025-12-31  
**Locations:** Hamburg, Berlin, Cologne, Frankfurt, and Munich  
**Final weather covariates:** mean 2 m air temperature and mean 2 m relative humidity across the five locations.

Only the two weather variables used by the final forecasting experiment are collected here. Wind speed and precipitation were present in an earlier exploratory notebook but were not used in the final model dataset.

The five-city mean is an equal-weighted spatial proxy for Germany-wide weather conditions. It is not a population-weighted or area-weighted national meteorological average.


In [ ]:

from datetime import datetime, timezone
from pathlib import Path
import json

import pandas as pd
import requests


## 1. Configuration

Edit `PROJECT_ROOT` only if the notebook is not being run from the thesis project root.

In [ ]:

# Resolve the repository root whether Jupyter starts in the project root
# or directly inside the notebooks/ directory.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR


START_DATE = '2015-01-01'
END_DATE = '2025-12-31'
TIMEZONE = 'UTC'
MODEL = 'era5_land'
API_URL = 'https://archive-api.open-meteo.com/v1/archive'

RAW_WEATHER_DIR = PROJECT_ROOT / 'data' / 'raw' / 'weather'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_WEATHER_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_WEATHER_FILE = PROCESSED_DIR / 'weather_germany_hourly_2015_2025.csv'
METADATA_FILE = PROCESSED_DIR / 'weather_collection_metadata.json'

CITIES = {
    'hamburg':   {'latitude': 53.5511, 'longitude': 9.9937},
    'berlin':    {'latitude': 52.5200, 'longitude': 13.4050},
    'cologne':   {'latitude': 50.9375, 'longitude': 6.9603},
    'frankfurt': {'latitude': 50.1109, 'longitude': 8.6821},
    'munich':    {'latitude': 48.1351, 'longitude': 11.5820},
}

HOURLY_VARIABLES = [
    'temperature_2m',
    'relative_humidity_2m',
]

print('Project root:', PROJECT_ROOT)
print('Final output:', FINAL_WEATHER_FILE)


## 2. Download function

In [ ]:

def download_city_weather(city_name: str, latitude: float, longitude: float) -> pd.DataFrame:
    """Download one city's hourly ERA5-Land weather data and return UTC timestamps."""
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': START_DATE,
        'end_date': END_DATE,
        'hourly': ','.join(HOURLY_VARIABLES),
        'timezone': TIMEZONE,
        'models': MODEL,
    }

    response = requests.get(API_URL, params=params, timeout=120)
    response.raise_for_status()
    payload = response.json()

    if 'hourly' not in payload:
        raise ValueError(f'Open-Meteo returned no hourly data for {city_name}: {payload}')

    city = pd.DataFrame(payload['hourly'])
    required = {'time', *HOURLY_VARIABLES}
    missing = required.difference(city.columns)
    if missing:
        raise ValueError(f'{city_name}: missing expected API fields: {sorted(missing)}')

    city['timestamp'] = pd.to_datetime(city['time'], utc=True, errors='raise')
    city = city.drop(columns='time').rename(columns={
        'temperature_2m': f'temperature_{city_name}',
        'relative_humidity_2m': f'humidity_{city_name}',
    })

    city = city.sort_values('timestamp').reset_index(drop=True)

    if city['timestamp'].duplicated().any():
        raise ValueError(f'{city_name}: duplicate UTC timestamps detected.')
    if city.isna().any().any():
        raise ValueError(f'{city_name}: missing weather values detected.\n{city.isna().sum()}')
    if not city['timestamp'].diff().dropna().eq(pd.Timedelta(hours=1)).all():
        raise ValueError(f'{city_name}: timestamp sequence is not strictly hourly.')

    return city


## 3. Collect and save city-level data

In [ ]:

city_frames = []

for city_name, coordinates in CITIES.items():
    print(f'Downloading {city_name.title()}...')
    city_df = download_city_weather(
        city_name,
        coordinates['latitude'],
        coordinates['longitude'],
    )
    city_frames.append(city_df)

    city_file = RAW_WEATHER_DIR / f'{city_name}_weather_2015_2025.csv'
    city_df.to_csv(city_file, index=False)
    print(f'  rows: {len(city_df):,} | saved: {city_file}')


## 4. Construct the five-city Germany weather proxy

In [ ]:

from functools import reduce

weather_wide = reduce(
    lambda left, right: pd.merge(left, right, on='timestamp', how='inner', validate='one_to_one'),
    city_frames,
)

temperature_columns = [f'temperature_{city}' for city in CITIES]
humidity_columns = [f'humidity_{city}' for city in CITIES]

weather_wide['temperature_mean'] = weather_wide[temperature_columns].mean(axis=1)
weather_wide['humidity_mean'] = weather_wide[humidity_columns].mean(axis=1)

weather = weather_wide[
    ['timestamp', 'temperature_mean', 'humidity_mean']
].copy()

weather = weather.sort_values('timestamp').reset_index(drop=True)
weather.head()


## 5. Data-quality checks

These assertions deliberately fail rather than silently imputing or deleting observations.

In [ ]:

expected_index = pd.date_range(
    start=f'{START_DATE} 00:00:00',
    end=f'{END_DATE} 23:00:00',
    freq='h',
    tz='UTC',
)

assert len(weather) == len(expected_index), (
    f'Unexpected row count: {len(weather):,}; expected {len(expected_index):,}.'
)
assert weather['timestamp'].equals(pd.Series(expected_index, name='timestamp')), (
    'Weather timestamps do not exactly match the expected continuous hourly UTC index.'
)
assert not weather['timestamp'].duplicated().any(), 'Duplicate timestamps detected.'
assert weather.isna().sum().sum() == 0, 'Missing values detected.'
assert weather['humidity_mean'].between(0, 100).all(), 'Humidity outside [0, 100]% detected.'
assert weather['temperature_mean'].between(-50, 60).all(), 'Implausible temperature detected.'

print('Shape:', weather.shape)
print('UTC range:', weather['timestamp'].min(), '→', weather['timestamp'].max())
print('\nMissing values:')
print(weather.isna().sum())
print('\nSummary statistics:')
print(weather[['temperature_mean', 'humidity_mean']].describe())


## 6. Save the final weather dataset and provenance metadata

In [ ]:

weather.to_csv(FINAL_WEATHER_FILE, index=False)

metadata = {
    'source': 'Open-Meteo Historical Weather API',
    'api_url': API_URL,
    'model': MODEL,
    'start_date': START_DATE,
    'end_date': END_DATE,
    'timezone': TIMEZONE,
    'hourly_variables': HOURLY_VARIABLES,
    'locations': CITIES,
    'aggregation': 'equal-weighted arithmetic mean across the five cities at each UTC hour',
    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
    'rows': int(len(weather)),
}

with METADATA_FILE.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Saved weather dataset:', FINAL_WEATHER_FILE)
print('Saved metadata:', METADATA_FILE)



## Output

The preprocessing notebook expects:

`data/processed/weather_germany_hourly_2015_2025.csv`

No interpolation, scaling, or future-weather construction is performed here.
